In [18]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col
from pyspark.sql.types import FloatType
import numpy as np
from models.fcm import Dfcm

In [2]:
# Initialize Spark session
spark = SparkSession.builder.appName("FCM_PySpark").getOrCreate()

24/08/23 12:39:17 WARN Utils: Your hostname, ubuntu resolves to a loopback address: 127.0.1.1; using 192.168.0.106 instead (on interface wlp6s0)
24/08/23 12:39:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/23 12:39:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [20]:
# Step 1: Load data from HDFS
csv_file_path ="data/csv/602_Dry_Bean.csv"
data = spark.read.csv(csv_file_path, header=True, inferSchema=True)

## Drop the last column (labels) 
data = data.drop(data.columns[-1])

## Convert all columns to float type    
for column in data.columns:
    data = data.withColumn(column, col(column).cast(FloatType()))

# Step 2: Prepare data for clustering
assembler = VectorAssembler(inputCols=data.columns, outputCol="features")
data_vectorized = assembler.transform(data)

In [47]:
# Convert to RDD
data_rdd = data_vectorized.select("features").rdd.map(lambda row: np.array(row["features"]))


In [48]:
# Step 3: Initialize the FCM model
fcm_model = Dfcm(m=2, epsilon=1e-5, maxiter=10000)

# Step 4: Run FCM algorithm
def run_fcm_partition(partition, C, seed):
    """
    Function to run FCM on a partition.
    """
    partition_data = np.array(list(partition))
    if len(partition_data) == 0:
        return iter([])
    u, v, iterations = fcm_model.cmeans(partition_data, C, seed)
    return iter([(u, v, iterations)])

# Number of clusters
C = 7
seed = 42

# Run FCM on RDD partitions
results_rdd = data_rdd.mapPartitions(lambda partition: run_fcm_partition(partition, C, seed))

# Collect results
results = results_rdd.collect()

# Combine results from all partitions (This is a simplified step; further merging logic might be needed)
U = np.vstack([result[0] for result in results])
V = np.mean([result[1] for result in results], axis=0)  # Averaging centroids
iterations = max([result[2] for result in results])

print(f"Final membership matrix (U): \n{U}")
print(f"Final cluster centers (V): \n{V}")
print(f"Total iterations: {iterations}")

# Stop Spark session
spark.stop()

Final membership matrix (U): 
[[1.27135867e-03 6.30108665e-04 3.64044008e-04 ... 1.18230513e-02
  5.66050633e-05 3.36270480e-03]
 [5.58269385e-04 2.74609883e-04 1.57975862e-04 ... 5.47391956e-03
  2.43627893e-05 1.50047442e-03]
 [2.74220216e-05 1.33372095e-05 7.62376372e-06 ... 2.92035174e-04
  1.16157347e-06 7.55253325e-05]
 ...
 [2.46909327e-02 8.10065939e-03 3.82361910e-03 ... 6.67999341e-01
  4.23768008e-04 2.46005385e-01]
 [2.52387462e-02 8.25907541e-03 3.89418326e-03 ... 6.58091804e-01
  4.30908222e-04 2.54758692e-01]
 [2.49550746e-02 8.17722601e-03 3.85776625e-03 ... 6.63252363e-01
  4.27229657e-04 2.50191591e-01]]
Final cluster centers (V): 
[[5.97238946e+04 9.64068509e+02 3.74340728e+02 2.05426158e+02
  1.84210399e+00 8.26062521e-01 6.06387508e+04 2.75590138e+02
  7.25268259e-01 9.84952706e-01 8.08150994e-01 7.39740739e-01
  6.28749661e-03 1.17262681e-03 5.50180416e-01 9.92751784e-01]
 [7.28905782e+04 1.05493911e+03 3.95420390e+02 2.36826375e+02
  1.67713069e+00 7.95148184e-01